# 第四章：分離資料和指令

- [課程](#lesson)
- [練習](#exercises)
- [範例練習場](#example-playground)

## 設置

執行以下設置 cell 以載入你的 API 金鑰並建立 `get_completion` 輔助函數。


In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## 課程

通常，我們不想寫完整的 prompt，而是想要**可以在提交給 Claude 之前用額外輸入資料修改的 prompt 模板**。這可能是因為 prompt 的某些部分是固定不變的，而其他部分是動態的。

**使用 XML tags 是分離資料和指令的最簡單方法之一**。


### 範例

在第一個範例中，我們要求 Claude 作為動物聲音生成器。請注意，提交給 Claude 的完整 prompt 只是用輸入（`ANIMAL`）替換了 `PROMPT_TEMPLATE`。


In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

為什麼我們想要這樣分離和替換輸入？**Prompt 模板簡化了重複性任務**。當你的 prompt 結構允許用戶提交內容時，你需要一個機制讓 Claude 知道哪些是你的指令，哪些是用戶的內容。


當引入像這樣的替換變數時，**確保 Claude 知道變數的開始和結束位置**非常重要（相對於指令或任務描述）。


In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

Claude 認為「Yo Claude」是它應該重寫的電子郵件的一部分！要解決這個問題，**用 XML tags 包裹輸入**！


我們使用 [XML tags](https://docs.anthropic.com/claude/docs/use-xml-tags) 來清楚區分指令和輸入資料。**XML tags 看起來像這樣：`<tag></tag>`**。

使用 XML tags 有幾個優點：
- 讓 Claude 清楚知道哪些是指令，哪些是資料
- 讓你更容易提取結構化輸出
- 提高 prompt 的整體可靠性


In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

讓我們看另一個 XML tags 如何幫助我們的範例。

在以下 prompt 中，**Claude 錯誤地解釋了哪部分是指令，哪部分是輸入**。


In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

要修復這個問題，只需**用 XML tags 包裹用戶輸入的句子**。


In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

**注意：** 在「關於動物」prompt 的錯誤版本中，我們必須在其中加入連字符，以便 Claude 以我們在這個例子中想要的錯誤方式回應。這是一個重要的提示——Claude 通常非常智能，能夠識別你的意圖，即使 prompt 的格式不完美。


---

## 練習
- [練習 4.1 - 俳句主題](#exercise-41---haiku-topic)
- [練習 4.2 - 有錯別字的狗問題](#exercise-42---dog-question-with-typos)
- [練習 4.3 - 狗問題第二部分](#exercise-43---dog-question-part-2)


### 練習 4.1 - 俳句主題
修改 `PROMPT` 使其成為一個模板，接受名為 `TOPIC` 的變數並輸出關於該主題的俳句。


In [ ]:
# Variable content
TOPIC = "Pigs"

# Prompt template with a placeholder for the variable content
PROMPT = f""

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("pigs", text.lower()) and re.search("haiku", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果你想要提示，執行下面的 cell！


In [ ]:
from hints import exercise_4_1_hint; print(exercise_4_1_hint)

### 練習 4.2 - 有錯別字的狗問題
通過添加 XML tags 修復 `PROMPT`，使 Claude 產生正確的答案。


In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果你想要提示，執行下面的 cell！


In [ ]:
from hints import exercise_4_2_hint; print(exercise_4_2_hint)

### 練習 4.3 - 狗問題第二部分
**不添加** XML tags 修復 `PROMPT`。相反，只從 prompt 中刪除一兩個字。


In [ ]:
# Variable content
QUESTION = "ar cn brown?"

# Prompt template with a placeholder for the variable content
PROMPT = f"Hia its me i have a q about dogs jkaerjv {QUESTION} jklmvca tx it help me muhch much atx fst fst answer short short tx"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search("brown", text.lower()))

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(response)
print("\n------------------------------------------ GRADING ------------------------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ 如果你想要提示，執行下面的 cell！


In [ ]:
from hints import exercise_4_3_hint; print(exercise_4_3_hint)

### 恭喜！

如果你完成了到目前為止的所有練習，你已準備好進入下一章。Happy prompting！


---

## 範例練習場

這是一個讓你自由實驗本課程中展示的 prompt 範例，並調整 prompt 以觀察它如何影響 Claude 回應的區域。


In [ ]:
# Variable content
ANIMAL = "Cow"

# Prompt template with a placeholder for the variable content
PROMPT = f"I will tell you the name of an animal. Please respond with the noise that animal makes. {ANIMAL}"

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. {EMAIL} <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
EMAIL = "Show up at 6am tomorrow because I'm the CEO and I say so."

# Prompt template with a placeholder for the variable content
PROMPT = f"Yo Claude. <email>{EMAIL}</email> <----- Make this email more polite but don't change anything else about it."

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f"""Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
{SENTENCES}"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))

In [ ]:
# Variable content
SENTENCES = """- I like how cows sound
- This sentence is about spiders
- This sentence may appear to be about dogs but it's actually about pigs"""

# Prompt template with a placeholder for the variable content
PROMPT = f""" Below is a list of sentences. Tell me the second item on the list.

- Each is about an animal, like rabbits.
<sentences>
{SENTENCES}
</sentences>"""

# Print Claude's response
print("--------------------------- Full prompt with variable substutions ---------------------------")
print(PROMPT)
print("\n------------------------------------- Claude's response -------------------------------------")
print(get_completion(PROMPT))